# Code Generator (Trình sinh mã)

Yêu cầu: dùng Frontier model (mô hình hàng đầu) để sinh mã C++ hiệu năng cao từ mã Python


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc nhở: lấy code mới nhất</h2>
            <span style="color:#f71;">Mình liên tục cải thiện các lab (bài thực hành), thêm ví dụ và bài tập.
            Đầu mỗi tuần, nên kiểm tra bạn đã có code mới nhất chưa.<br/>
            Trước hết hãy <a href="https://chatgpt.com/share/6734e705-3270-8012-a074-421661af6ba9">git pull và merge (gộp) thay đổi của bạn nếu cần</a>. Gặp vấn đề? Hỏi ChatGPT cách merge — hoặc liên hệ mình!<br/><br/>
            Sau khi pull code, từ thư mục llm_engineering, trong Cursor Terminal, chạy:<br/>
            <code>uv sync</code><br/>
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Lưu ý quan trọng</h1>
            <span style="color:#900;">
            Trong lab (bài thực hành) này, mình dùng các model cao cấp GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4 — những model có giá hơi cao hơn. Chi phí vẫn thấp, nhưng nếu bạn muốn giữ chi phí cực thấp, hãy chọn model rẻ hơn như gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# imports (các thư viện)

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("Chưa thiết lập OpenAI API Key")
    
if anthropic_api_key:
    print(f"Anthropic API Key tồn tại và bắt đầu bằng {anthropic_api_key[:7]}")
else:
    print("Chưa thiết lập Anthropic API Key (và đây là tùy chọn)")

if google_api_key:
    print(f"Google API Key tồn tại và bắt đầu bằng {google_api_key[:2]}")
else:
    print("Chưa thiết lập Google API Key (và đây là tùy chọn)")

if grok_api_key:
    print(f"Grok API Key tồn tại và bắt đầu bằng {grok_api_key[:4]}")
else:
    print("Chưa thiết lập Grok API Key (và đây là tùy chọn)")

In [ ]:
# Kết nối các client library (thư viện client)

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)

In [ ]:
OPENAI_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
GROK_MODEL = "grok-4"
GEMINI_MODEL = "gemini-2.5-pro"

# Muốn giữ chi phí cực thấp? Bỏ comment (dòng chú thích) các dòng sau:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-haiku-4-5"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-3.1-flash-lite"

## LƯU Ý:

Chúng ta sẽ viết giải pháp chuyển Python thành mã C++ hiệu quả, đã tối ưu (optimized) cho máy của bạn, rồi biên dịch thành native machine code (mã máy gốc) và chạy.

Bạn không bắt buộc phải tự chạy code — đó không phải mục tiêu chính của bài tập!

Nhưng nếu muốn (vì khá thú vị!), mình ghi các bước ở đây. Hoàn toàn tùy chọn!

Ngoài ra, mình cũng sẽ chỉ một website để bạn chạy mã C++.

In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

In [ ]:
message = f"""
Đây là báo cáo thông tin hệ thống (system information) của máy tính tôi.
Tôi muốn chạy C++ compiler (trình biên dịch C++) để biên dịch một file C++ tên main.cpp rồi thực thi theo cách đơn giản nhất.
Hãy trả lời xem tôi có cần cài C++ compiler nào không. Nếu có, hãy đưa hướng dẫn từng bước đơn giản nhất.

Nếu máy tôi đã sẵn sàng biên dịch C++, tôi muốn chạy đoạn Python tương tự như sau để biên dịch và thực thi:
```python
compile_command = # điền lệnh ở đây — để đạt runtime performance (hiệu năng khi chạy) nhanh nhất có thể
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # điền lệnh ở đây
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Hãy cho tôi chính xác nên dùng gì cho compile_command và run_command.

Thông tin hệ thống:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

## Nếu bạn cần cài thêm phần mềm

Nếu muốn, hãy làm theo hướng dẫn của GPT! Sau đó chạy lại phần phân tích (có thể cần Restart notebook) để xác nhận đã sẵn sàng.

Bạn sẽ có lệnh biên dịch code và lệnh chạy chương trình!

Điền các lệnh đó vào ô bên dưới:

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

## Tiếp theo: nhiệm vụ chính

In [ ]:
system_prompt = """
Nhiệm vụ của bạn là chuyển mã Python thành mã C++ hiệu năng cao (high performance).
Chỉ trả lời bằng mã C++. Không giải thích, trừ một vài comment (chú thích) khi cần.
Mã C++ phải cho ra output (kết quả in ra) giống hệt, trong thời gian ngắn nhất có thể.
"""

def user_prompt_for(python):
    return f"""
Port (chuyển) mã Python này sang C++ với implementation (cách hiện thực) nhanh nhất, cho ra output giống hệt trong thời gian ngắn nhất.
Thông tin hệ thống là:
{system_info}
Phản hồi của bạn sẽ được ghi vào file tên main.cpp rồi biên dịch và thực thi; lệnh compilation (biên dịch) là:
{compile_command}
Chỉ trả lời bằng mã C++.
Mã Python cần port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [ ]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [ ]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Kết quả (Result): {result:.12f}")
print(f"Thời gian thực thi (Execution Time): {(end_time - start_time):.6f} giây")
"""

In [ ]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [ ]:
run_python(pi)

In [ ]:
port(openai, OPENAI_MODEL, pi)

# Biên dịch C++ và thực thi

Ô tiếp theo chứa lệnh biên dịch file C++ dựa trên hướng dẫn từ GPT.

Một lần nữa, bước này không bắt buộc nếu bạn không muốn!

HOẶC cách khác: học viên Sandeep K.G. gợi ý có thể chạy Python và C++ online để thử. Cảm ơn Sandeep!  
> Không phải so sánh chính xác tuyệt đối, nhưng vẫn thấy được chênh lệch performance (hiệu năng).  
> Ví dụ tại: https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
# Dùng các lệnh từ GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [ ]:
compile_and_run()

In [ ]:
19.178207/0.082168

## Được rồi, thử các model còn lại!

In [ ]:
port(anthropic, CLAUDE_MODEL, pi)
compile_and_run()

In [ ]:
port(grok, GROK_MODEL, pi)
compile_and_run()

In [ ]:
port(gemini, GEMINI_MODEL, pi)
compile_and_run()


In [ ]:
print(f"""
Trong thí nghiệm của Ed, các mức tăng tốc (performance speedup) là:

Hạng 4: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup (tăng tốc)
Hạng 3: GPT-5: {19.178207/0.082168:.0f}X speedup (tăng tốc)
Hạng 2: Grok 4: {19.178207/0.018092:.0f}X speedup (tăng tốc)
Hạng 1: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup (tăng tốc)
""")